<a href="https://colab.research.google.com/github/SharaniS/notes_generator/blob/main/notes_generate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q \
    openai-whisper \
    ffmpeg-python \
    transformers \
    sentencepiece \
    accelerate \
    torch
!pip install -q transformers accelerate torch
!pip install -q gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import os
import re
import torch
import whisper
import ffmpeg
import gradio as gr

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
def extract_audio(video_path, audio_path):
    (
        ffmpeg
        .input(video_path)
        .output(audio_path, ac=1, ar=16000)
        .overwrite_output()
        .run(quiet=True)
    )

In [5]:
video_path = "/content/drive/MyDrive/lecture_project/nptel_lecture.mp4"
audio_path = "/content/drive/MyDrive/lecture_project/lecture.wav"

extract_audio(video_path, audio_path)
print("Audio extracted successfully!")

Audio extracted successfully!


In [6]:
whisper_model = whisper.load_model("base")  # or "small"

def transcribe_audio(audio_path):
    result = whisper_model.transcribe(audio_path)
    return result["text"]

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 180MiB/s]


In [7]:
raw_transcript = transcribe_audio(audio_path)

with open("/content/drive/MyDrive/lecture_project/raw.txt", "w") as f:
    f.write(raw_transcript)

print("Raw transcript saved successfully!")

Raw transcript saved successfully!


In [8]:
def clean_transcript(text):
    text = text.lower()
    text = re.sub(r"\b(uh|um|you know|like)\b", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

cleaned_transcript = clean_transcript(raw_transcript)

with open("/content/drive/MyDrive/lecture_project/cleaned.txt", "w") as f:
    f.write(cleaned_transcript)

print("Transcript cleaned!")

Transcript cleaned!


In [9]:
def chunk_text(text, max_words=180):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunks.append(" ".join(words[i:i+max_words]))
    return chunks

chunks = chunk_text(cleaned_transcript)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 3


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ Gemma loaded successfully!")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✅ Gemma loaded successfully!


In [12]:
def generate_notes(text):

    prompt = f"""
Create structured study notes from this lecture.

Lecture:
{text}

Format:

Title:
Key Concepts:
Definitions:
Examples:
Important Points:
Summary:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=320,      # ✅ Prevent summary cutoff
        do_sample=False,         # ✅ More stable structure
        eos_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    notes = decoded.replace(prompt, "").strip()

    return notes


def clean_notes(notes):

    notes = notes.replace("Exercises:", "")
    notes = notes.replace("Additional Reading:", "")
    notes = notes.replace("****", "")
    notes = notes.replace("**", "")      # ✅ Remove markdown stars

    lines = notes.split("\n")
    filtered = []

    for line in lines:
        if line.strip().startswith("* What"):
            continue
        if line.strip().startswith("* Why"):
            continue
        # New filter: Remove lines starting with a number and a dot, typically exercise-like
        if re.match(r"^\d+\.", line.strip()):
            continue
        filtered.append(line)

    return "\n".join(filtered).strip()


# ✅ RUN GENERATION
notes = generate_notes(cleaned_transcript[:500])

# ✅ CLEAN OUTPUT
cleaned_notes = clean_notes(notes)

# ✅ SAVE TO DRIVE
with open("/content/drive/MyDrive/lecture_project/notes.txt", "w") as f:
    f.write(cleaned_notes)

print("✅ Notes saved")

# ✅ DISPLAY
print("\nCLEANED NOTES:\n")
print(cleaned_notes)
print("\nLength:", len(cleaned_notes))

✅ Notes saved

CLEANED NOTES:

Key Concepts:

* Class: A blueprint for creating objects.
* Object: An instance of a class.
* Encapsulation: Hiding data and methods of an object from outside entities.

Definitions:

* Data: The information or value stored in an object.
* Methods: The actions or operations that can be performed on an object.

Examples:

* A car class can have data members like `color` and `model` and methods like `start` and `stop`.
* A student class can have data members like `name` and `age` and methods like `study` and `learn`.

Important Points:

* Encapsulation protects data and methods from unauthorized access.
* It allows us to control the access to data and methods, making it easier to maintain and modify the code.
* Encapsulation is a fundamental concept in object-oriented programming.

Summary:

Encapsulation is a mechanism that hides data and methods of an object from outside entities. It allows us to control the access to data and methods, making it easier to

In [25]:
import re

def generate_quiz_from_transcript(transcript):

    prompt = f"""
Generate EXACTLY 5 multiple choice theory questions from the transcript below.

STRICT RULES:
- Follow the format EXACTLY
- No explanations
- No markdown
- No bold text
- Start directly with Q1
- Each question must have 4 options
- Provide answer in format: Ans: a/b/c/d

FORMAT:

Q1. Question text
a) Option
b) Option
c) Option
d) Option
Ans: x

Transcript:
{transcript}
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=700,      # Increased max_new_tokens
        do_sample=False,          # ✅ deterministic
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    input_len = inputs["input_ids"].shape[1]

    decoded = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    return clean_quiz_output(decoded)


def clean_quiz_output(text):

    lines = [l.strip() for l in text.split("\n") if l.strip()]

    questions = []
    block = []

    for line in lines:
        if line.startswith("Q"):
            if block:
                questions.append(block)
            block = [line]
        else:
            block.append(line)

    if block:
        questions.append(block)

    cleaned = []

    for i, q in enumerate(questions[:5], 1):

        first = re.sub(r"^Q\d+\.", f"Q{i}.", q[0])

        opts = [l for l in q if re.match(r"^[a-d]\)", l)]
        ans = [l for l in q if l.startswith("Ans:")]

        if len(opts) == 4 and ans:
            cleaned.append("\n".join([first] + opts + [ans[0]]))

    return "\n\n".join(cleaned)

In [14]:
import re

def generate_flashcards(transcript):

    prompt = f"""
Generate 5 flashcards from the text below. Each flashcard should have a Question and an Answer.

Text:
{transcript}

Strictly follow this format for each flashcard:
Question: [Your question based on the text]
Answer: [Your answer based on the text]

Flashcards:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500, # Increased max_new_tokens
        do_sample=True,           # Re-enable sampling
        temperature=0.7,          # Allow some creativity
        top_p=0.9,                # Focus on likely tokens
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    input_length = inputs["input_ids"].shape[1]

    decoded = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    # --- Robust Parsing --- General method to extract Q&A pairs
    flashcards_list = []
    lines = [line.strip() for line in decoded.split("\n") if line.strip()]

    current_question = None
    current_answer = None

    for line in lines:
        # Check for 'Question:' possibly preceded by a number (e.g., '1. Question:')
        q_match = re.match(r"^\d*\.?\s*Question:\s*(.*)", line)
        if q_match:
            if current_question and current_answer:
                flashcards_list.append({"Question": current_question, "Answer": current_answer})
                current_question = None # Reset for the new card
                current_answer = None
            current_question = q_match.group(1).strip()
        # Check for 'Answer:'
        a_match = re.match(r"^Answer:\s*(.*)", line)
        if a_match:
            current_answer = a_match.group(1).strip()

        if len(flashcards_list) == 5: # Stop if we already have 5 flashcards
            break

    # Add the last parsed card if it's complete and we haven't reached 5 yet
    if current_question and current_answer and len(flashcards_list) < 5:
        flashcards_list.append({"Question": current_question, "Answer": current_answer})

    return flashcards_list[:5] # Ensure max 5 flashcards are returned

# Call the function and print the results
generated_flashcards = generate_flashcards(cleaned_notes)

print("\n------ GENERATED FLASHCARDS ------\n")

if generated_flashcards:
    for i, card in enumerate(generated_flashcards, 1):
        print(f"Flashcard {i}:")
        print(f"Question: {card['Question']}")
        print(f"Answer: {card['Answer']}\n")
else:
    print("No flashcards were generated. The model might not have followed the format.")



------ GENERATED FLASHCARDS ------

Flashcard 1:
Question: What is the difference between data and methods?
Answer: Data is information stored in an object, while methods are actions or operations that can be performed on an object.

Flashcard 2:
Question: How does encapsulation protect data and methods?
Answer: Encapsulation protects data and methods from unauthorized access by hiding them from outside entities.

Flashcard 3:
Question: What is an example of data?
Answer: Data can include variables, constants, and objects.

Flashcard 4:
Question: What is an example of a method?
Answer: Methods can include functions, subroutines, and constructors.

Flashcard 5:
Question: What is the importance of encapsulation in object-oriented programming?
Answer: Encapsulation is a fundamental concept in object-oriented programming and provides a mechanism for controlling access to data and methods, making it easier to maintain and modify the code.



In [34]:
def process_lecture(video_file, video_path):

    import tempfile
    import os

    if video_file is None and not video_path:
        return "No video provided", "", "", ""

    # ✅ Handle upload safely
    if video_file is not None:
        video_input = video_file.name   # Gradio already gives temp file
    else:
        video_input = video_path.strip()

    audio_temp = "temp_audio.wav"

    try:
        print("Extracting audio...")
        extract_audio(video_input, audio_temp)

        print("Transcribing...")
        transcript = transcribe_audio(audio_temp)

        print("Cleaning transcript...")
        cleaned = clean_transcript(transcript)
        print(f"Cleaned transcript length (before quiz gen): {len(cleaned)}") # NEW DEBUG
        print(f"Cleaned transcript (first 200 chars): {cleaned[:200]}") # NEW DEBUG

        print("Generating notes...")
        notes_raw = generate_notes(cleaned[:500])
        notes = clean_notes(notes_raw)

        print("Generating quiz...")
        raw_quiz_output = generate_quiz_from_transcript(cleaned)
        print(f"Raw Quiz Model Output (before cleaning):\n{raw_quiz_output}") # Debug print: Print full raw output
        quiz = clean_quiz_output(raw_quiz_output)
        print(f"Quiz content length: {len(quiz)}") # Added debug print back
        print(f"Quiz content (first 200 chars): {quiz[:200]}") # Added debug print back

        print("Generating flashcards...")
        flashcards_list = generate_flashcards(notes)

        flashcards_text = ""
        for i, card in enumerate(flashcards_list, 1):
            flashcards_text += f"Flashcard {i}\n"
            flashcards_text += f"Q: {card['Question']}\n"
            flashcards_text += f"A: {card['Answer']}\n\n"

        return cleaned, notes, quiz, flashcards_text

    except Exception as e:
        return f"Error: {str(e)}", "", "", ""

    finally:
        if os.path.exists(audio_temp):
            os.remove(audio_temp)

In [ ]:
import gradio as gr
import asyncio # Import asyncio

with gr.Blocks(title="AI Lecture Assistant") as ui:

    gr.Markdown("# 🎓 AI Lecture Assistant")
    gr.Markdown("Generate transcript, notes, quiz, and flashcards from lecture videos.")

    with gr.Row():
        video_file = gr.File(label="Upload Lecture Video")
        video_path = gr.Textbox(label="OR Enter Video Path")

    submit_btn = gr.Button("Submit")

    # ✅ Output Sections (defined ONLY once)
    with gr.Accordion("📜 Transcript", open=False):
        transcript_out = gr.Textbox(lines=10)

    with gr.Accordion("📝 Notes", open=False):
        notes_out = gr.Textbox(lines=10)

    with gr.Accordion("❓ Quiz", open=False):
        quiz_out = gr.Textbox(lines=10)

    with gr.Accordion("🧠 Flashcards", open=False):
        flashcards_out = gr.Textbox(lines=10)

    submit_btn.click(
        fn=process_lecture,
        inputs=[video_file, video_path],
        outputs=[transcript_out, notes_out, quiz_out, flashcards_out]
    )

# Explicitly set the event loop policy to avoid RuntimeError in some environments
asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())

ui.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ea80acf7fad43ba685.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Extracting audio...
Transcribing...
Cleaning transcript...
Cleaned transcript length (before quiz gen): 1725
Cleaned transcript (first 200 chars): today, we shall discuss about a very important object oriented concept. it is called encapsulation. so, our today's topic is encapsulation in java. as we know, the concept of class is a basic things i
Generating notes...
Generating quiz...
Raw Quiz Model Output (before cleaning):

Quiz content length: 0
Quiz content (first 200 chars): 
Generating flashcards...


In [27]:
print("\n--- Testing Quiz Generation Directly ---\n")

# Ensure cleaned_transcript is available (it should be from previous runs)
if 'cleaned_transcript' in globals():
    test_transcript = cleaned_transcript
    print("Using existing 'cleaned_transcript' for testing.")
else:
    test_transcript = "This is a short test transcript about object oriented programming and encapsulation. It explains classes, objects, methods, and data members." # Fallback if not available
    print("Warning: 'cleaned_transcript' not found, using a dummy transcript for testing.")

# Generate raw quiz output
test_raw_quiz_output = generate_quiz_from_transcript(test_transcript)
print("\nRaw Quiz Model Output (before cleaning):\n")
print(test_raw_quiz_output)
print(f"\nRaw Quiz content length: {len(test_raw_quiz_output)}")

# Clean the quiz output
test_cleaned_quiz = clean_quiz_output(test_raw_quiz_output)
print("\nCleaned Quiz Output:\n")
print(test_cleaned_quiz)
print(f"\nCleaned Quiz content length: {len(test_cleaned_quiz)}")


--- Testing Quiz Generation Directly ---

Using existing 'cleaned_transcript' for testing.

Raw Quiz Model Output (before cleaning):

Q1. Which is the most important element in a class?
a) Class
b) Field
c) Method
d) Constructor
Ans: b

Q2. Which is the most important element in a class that can be accessed from anywhere in the program?
a) Class
b) Field
c) Method
d) Constructor
Ans: a

Q3. Which is the most important element in a class that is responsible for creating and initializing an object?
a) Class
b) Field
c) Method
d) Constructor
Ans: a

Q4. Which is the most important element in a class that is responsible for defining the behavior of an object?
a) Class
b) Field
c) Method
d) Constructor
Ans: c

Raw Quiz content length: 579

Cleaned Quiz Output:

Q1. Which is the most important element in a class?
a) Class
b) Field
c) Method
d) Constructor
Ans: b

Q2. Which is the most important element in a class that can be accessed from anywhere in the program?
a) Class
b) Field
c) Method